# Biología Computacional — BC02

- **Módulo:** BC02
- **Sesión(es):** S2
- **Tema:** Bioquímica esencial como modelo computable
- **Asignatura:** Biología Computacional

Este cuaderno acompaña la práctica del módulo BC02. Ejecuta las celdas en orden en
[Google Colab](https://colab.research.google.com/).

**Pregunta guía:** ¿Qué información se pierde cuando una molécula biológica se convierte en un modelo computable?

## 1. Configuración del entorno

Verificamos que las herramientas necesarias están disponibles.

In [ ]:
!apt-get update > /dev/null 2>&1; apt-get install -y coreutils > /dev/null 2>&1
import os, platform, hashlib
print(f'Python: {platform.python_version()}')
print(f'Sistema: {platform.system()} {platform.release()}')
print(f'Entorno listo ✅')

## 2. Descarga del paquete datos

Descargamos el paquete starter desde GitHub.

In [ ]:
import urllib.request
import tarfile
import os

URL = 'https://github.com/Asn4code/pid_2026/raw/desarrollo/materiales/biologia-computacional/BC02/paquete/BC02_starter_v0.1.0.tar.gz'

if not os.path.exists('BC02_starter_v0.1.0.tar.gz'):
    print('Descargando paquete...')
    urllib.request.urlretrieve(URL, 'BC02_starter_v0.1.0.tar.gz')
    print('Descarga completa ✅')
else:
    print('Paquete ya descargado ✅')

## 3. Extracción y verificación

Descomprimimos los datos y verificamos que la descarga no se corrompió.

In [ ]:
if os.path.exists('BC02_starter_v0.1.0.tar.gz'):
    with tarfile.open('BC02_starter_v0.1.0.tar.gz') as tar:
        tar.extractall()
    print('Datos extraídos ✅')
    
    import glob
    print('\nArchivos disponibles:')
    for f in sorted(glob.glob('datos/*')):
        size = os.path.getsize(f)
        print(f'  {f} ({size} bytes)')
    for f in sorted(glob.glob('datos/corpus/*')):
        size = os.path.getsize(f)
        print(f'  {f} ({size} bytes)')

## 4. Verificación con checksums

Comparamos los checksums SHA-256 para garantizar integridad de los datos del módulo.

In [ ]:
import subprocess
import hashlib
import os

# Verificar mini.tsv contra SHA256SUMS
result = subprocess.run(['sha256sum', 'datos/mini.tsv'], capture_output=True, text=True)
actual_hash = result.stdout.split()[0]
expected_hash = None
with open('datos/SHA256SUMS') as f:
    for line in f:
        h, name = line.strip().split()
        if name == 'mini.tsv':
            expected_hash = h
print(f'mini.tsv: SHA-256 = {actual_hash}')
print(f'          Esperado = {expected_hash}')
match = actual_hash == expected_hash
print(f'          Estado: {"OK ✅" if match else "FALLO ❌"}')

# Verificar MANIFEST completo (rutas relativas a datos/, por eso cwd='datos')
result_m = subprocess.run(['sha256sum', '-c', 'MANIFEST.sha256'], capture_output=True, text=True, cwd='datos')
lines_out = result_m.stdout.strip().split('\n')
passed = sum(1 for l in lines_out if ': OK' in l)
failed = sum(1 for l in lines_out if ': FAILED' in l)
print(f'\nMANIFEST: {passed} OK, {failed} fallidos')

## 5. Exploración del caso mínimo

Observa la tabla de 7 aminoácidos. Antes de ejecutar, predice: ¿qué columnas esperas ver y qué representa cada una?

In [ ]:
import csv

print('=== mini.tsv ===')
print('Aminoácidos esenciales como modelo tabular')
print('-' * 80)
with open('datos/mini.tsv') as f:
    reader = csv.reader(f, delimiter='\t')
    header = next(reader)
    print('Columnas:', ' | '.join(header))
    print('-' * 80)
    for row in reader:
        print('  ' + ' | '.join(row))
print('-' * 80)
print('\n=== Tu predicción ===')
print('# Antes de ejecutar: ¿qué columnas esperabas y qué representa cada una?')

## 6. Análisis del caso de representaciones

El archivo `caso_s02.tsv` muestra 6 representaciones de un mismo aminoácido. Para cada una, reflexiona: ¿qué información se retiene y cuál se pierde?

In [ ]:
import csv

print('=== caso_s02.tsv ===')
print('Representación | Ejemplo | Información retenida | Información perdida')
print('-' * 90)
with open('datos/caso_s02.tsv') as f:
    reader = csv.reader(f, delimiter='\t')
    for i, row in enumerate(reader):
        if i == 0:
            continue  # encabezado
        representacion, ejemplo, retenida, perdida = row
        print(f'\nRepresentación: {representacion}')
        print(f'  Ejemplo: {ejemplo}')
        print(f'  Información retenida: {retenida}')
        print(f'  Información perdida: {perdida}')
        print(f'  ¿En qué contexto usarías esta representación?')

## 7. Cálculo de masa de un péptido

Lee `datos/corpus/file_enlace_peptidico.md` y calcula la masa de la secuencia GAVSK. Recuerda: la masa de un péptido no es solo la suma de residuos — se añade 18.015 Da por el agua del enlace peptídico.

In [ ]:
import csv
from decimal import Decimal, ROUND_HALF_UP

# Construir diccionario masa -> código a partir de mini.tsv (Decimal evita errores de redondeo en coma flotante)
masas = {}
with open('datos/mini.tsv') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        masas[row['codigo']] = Decimal(row['masa'])

secuencia = 'GAVSK'
AGUA = Decimal('18.015')  # masa de una molécula de agua (enlace peptídico)

suma = sum(masas[aa] for aa in secuencia)
masa_peptido = suma + AGUA
masa_redondeada = masa_peptido.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

print(f'Residuos de {secuencia}:')
for aa in secuencia:
    print(f'  {aa}: {masas[aa]} Da')
print(f'\nSuma de residuos: {suma} Da')
print(f'Masa del péptido (suma + {AGUA} Da de agua): {masa_peptido} Da')
print(f'Masa redondeada a 2 decimales: {masa_redondeada} Da')
print(f'\nResultado esperado (corpus): ≈ 550.61 Da')

# Comprobación
assert masa_redondeada == Decimal('550.61'), 'La masa calculada no coincide con la esperada'
print('Cálculo verificado ✅')


## 8. Reflexión final

Responde a las preguntas clave antes de entregar:

In [ ]:
print('=== Preguntas de reflexión ===')
print()
print('1. ¿Qué representación de un aminoácido usarías en una secuencia de proteína y por qué?')
print('   -> Escribe tu respuesta aquí:')
print()
print('2. ¿Qué información biológica se pierde al reducir un aminoácido a una fila de una tabla?')
print('   -> Escribe tu respuesta aquí:')
print()
print('3. ¿Por qué la masa de un péptido no es simplemente la suma de las masas de sus residuos?')
print('   -> Escribe tu respuesta aquí:')
print()
print('4. ¿Qué información se pierde cuando una molécula biológica se convierte en un modelo computable?')
print('   -> Escribe tu respuesta aquí:')